# OVR Logistic Regression with Thresholding

This notebook builds a threshold-based one-vs-rest logistic regression benchmark for ACC skill tagging.

Workflow:
- load ACC job, course, and skill embeddings from `../embedding/acc`
- join each embedded entity to its ground-truth `extracted_skills` label from the CSVs
- build the `jobs_plus_courses` dataset variant
- train one logistic regression model per skill on the train split
- tune thresholds on the validation split only
- select the final configuration using validation metrics
- report the final locked metrics on the held-out test split

Threshold tuning is kept on validation, while the final result is reported on test so that the final numbers reflect held-out performance rather than validation-tuned performance.


In [43]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

EMBEDDINGS_PATH = Path("../embedding/acc")
JOB_LABEL_CSV = Path("../data/acc/audit_tax_accounting_jobs.csv")
COURSE_LABEL_CSV = Path("../data/acc/acc_courses.csv")
SKILLS_EMBEDDINGS_PATH = EMBEDDINGS_PATH / "acc_skills_embeddings.jsonl"
JOBS_EMBEDDINGS_PATH = EMBEDDINGS_PATH / "acc_jobs_embeddings.jsonl"
COURSES_EMBEDDINGS_PATH = EMBEDDINGS_PATH / "acc_courses_embeddings.jsonl"

skills_embeddings = pd.read_json(SKILLS_EMBEDDINGS_PATH, lines=True)
jobs_embeddings = pd.read_json(JOBS_EMBEDDINGS_PATH, lines=True)
courses_embeddings = pd.read_json(COURSES_EMBEDDINGS_PATH, lines=True)
job_labels = pd.read_csv(JOB_LABEL_CSV)
course_labels = pd.read_csv(COURSE_LABEL_CSV)

print("Embedding splits:")
print(jobs_embeddings["split"].value_counts().sort_index().rename("jobs"))
print()
print(courses_embeddings["split"].value_counts().sort_index().rename("courses"))
print()
print(f"num_skills: {len(skills_embeddings)}")

Embedding splits:
split
test     20
train    59
val      19
Name: jobs, dtype: int64

split
test     13
train    38
val      13
Name: courses, dtype: int64

num_skills: 229


## Helper Functions

This section defines the repeated setup functions used throughout the notebook. These helpers:
- convert `extracted_skills` text into Python lists
- join embeddings to labels
- build binary label matrices
- train one logistic regression model per skill
- tune thresholds and compute metrics

Keeping these helpers in one place makes the main experiment cells shorter and easier to follow.


In [44]:
def parse_skill_list(value):
    """Parses a pipe-separated skill list string into a list of individual skills."""
    if pd.isna(value) or str(value).strip() == "":
        return []
    return [skill.strip() for skill in str(value).split("|") if skill.strip()]

def attach_actual_skill_lists(df, labels_df, left_key, right_key):
    """Merges the embeddings DataFrame with the labels DataFrame, attaches actual skill lists, and handles missing labels."""
    merged = df.merge(
        labels_df,
        left_on=left_key,
        right_on=right_key,
        how="left",
        suffixes=("_embed", "_label"),
    )

    missing_mask = merged["extracted_skills"].isna()
    dropped_ids = merged.loc[missing_mask, left_key].tolist()
    if dropped_ids:
        print(f"Dropping {len(dropped_ids)} unlabeled rows for {left_key}: {dropped_ids}")
        merged = merged.loc[~missing_mask].copy()

    merged["actual_skill_lists"] = merged["extracted_skills"].apply(parse_skill_list)
    merged["display_title"] = merged["title_label"] if "title_label" in merged.columns else merged["title"]
    merged["entity_id"] = merged[left_key]

    return merged[["entity_type", "entity_id", "display_title", "embedding", "actual_skill_lists"]].copy()

def build_indicator_matrix(skill_lists, skill_names):
    """Converts a list of skill lists into a binary indicator matrix based on the provided skill names."""
    skill_to_idx = {skill: idx for idx, skill in enumerate(skill_names)}
    indicator = np.zeros((len(skill_lists), len(skill_names)), dtype=np.uint8)

    for row_idx, skills in enumerate(skill_lists):
        for skill in skills:
            if skill in skill_to_idx:
                indicator[row_idx, skill_to_idx[skill]] = 1

    return indicator

def build_dataset(train_df, val_df, test_df, skill_names):
    """Builds the dataset by converting embeddings to numpy arrays and creating indicator matrices for the skill labels."""
    X_train = np.vstack(train_df["embedding"].to_numpy()).astype(np.float32)
    X_val = np.vstack(val_df["embedding"].to_numpy()).astype(np.float32)
    X_test = np.vstack(test_df["embedding"].to_numpy()).astype(np.float32)

    Y_train = build_indicator_matrix(train_df["actual_skill_lists"].tolist(), skill_names)
    Y_val = build_indicator_matrix(val_df["actual_skill_lists"].tolist(), skill_names)
    Y_test = build_indicator_matrix(test_df["actual_skill_lists"].tolist(), skill_names)

    return X_train, X_val, X_test, Y_train, Y_val, Y_test

def fit_ovr_probabilities(X_train, Y_train, X_val, skill_names):
    """Fits one-vs-rest logistic regression models for each skill and returns predicted probabilities for train and validation sets."""
    # Fit on train and score validation only; test is handled after model selection.
    train_proba = np.zeros((X_train.shape[0], len(skill_names)), dtype=np.float32)
    val_proba = np.zeros((X_val.shape[0], len(skill_names)), dtype=np.float32)

    fitted_mask = np.zeros(len(skill_names), dtype=bool)
    fitted_models = [None] * len(skill_names)
    constant_proba = np.full(len(skill_names), np.nan, dtype=np.float32)
    skipped_skills = []

    for j, skill_name in enumerate(skill_names):
        y_train_j = Y_train[:, j]
        unique_classes = np.unique(y_train_j)

        # LogisticRegression needs both classes; skipped skills keep constant probabilities for alignment.
        if len(unique_classes) < 2:
            constant_val = float(unique_classes[0])
            train_proba[:, j] = constant_val
            val_proba[:, j] = constant_val
            constant_proba[j] = constant_val
            skipped_skills.append(skill_name)
            continue

        model = LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42,
        )
        model.fit(X_train, y_train_j)

        train_proba[:, j] = model.predict_proba(X_train)[:, 1]
        val_proba[:, j] = model.predict_proba(X_val)[:, 1]

        fitted_mask[j] = True
        fitted_models[j] = model

    return train_proba, val_proba, fitted_mask, fitted_models, skipped_skills, constant_proba

def predict_ovr_probabilities(X, fitted_models, constant_proba, skill_names):
    """Predicts probabilities for each skill using the fitted one-vs-rest logistic regression models, handling any skills that were skipped during training."""
    proba = np.zeros((X.shape[0], len(skill_names)), dtype=np.float32)
    for j, model in enumerate(fitted_models):
        if model is None:
            if not np.isnan(constant_proba[j]):
                proba[:, j] = constant_proba[j]
            continue
        proba[:, j] = model.predict_proba(X)[:, 1]
    return proba

def micro_metrics(y_true, y_pred):
    """Calculates micro-averaged precision, recall, and F1 score for binary predictions."""
    tp = np.logical_and(y_pred == 1, y_true == 1).sum()
    fp = np.logical_and(y_pred == 1, y_true == 0).sum()
    fn = np.logical_and(y_pred == 0, y_true == 1).sum()

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

    return precision, recall, f1

def macro_metrics(y_true, y_pred):
    """Calculates macro-averaged precision, recall, and F1 score for binary predictions."""
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
    return precision, recall, f1

def tune_global_threshold(y_true, y_score):
    """Tunes a global threshold for converting predicted probabilities into binary predictions by maximizing micro-averaged F1 score."""
    candidate_thresholds = np.unique(y_score)
    # The extra value lets the search choose a threshold that predicts no positives.
    candidate_thresholds = np.r_[candidate_thresholds, 1.0 + 1e-9]

    best_threshold = 0.5
    best_candidate = None

    for threshold in candidate_thresholds:
        y_pred = (y_score >= threshold).astype(np.uint8)
        precision, recall, f1 = micro_metrics(y_true, y_pred)
        candidate = (f1, recall, -threshold)

        if best_candidate is None or candidate > best_candidate:
            best_candidate = candidate
            best_threshold = float(threshold)

    return best_threshold

def tune_best_threshold_for_one_skill(y_true, y_score):
    """Tunes the best threshold for a single skill by maximizing F1 score, with recall as a tiebreaker and preferring higher thresholds in case of ties."""
    candidate_thresholds = np.unique(y_score)
    candidate_thresholds = np.r_[candidate_thresholds, 1.0 + 1e-9]

    best_threshold = 0.5
    best_candidate = None

    for threshold in candidate_thresholds:
        y_pred = (y_score >= threshold).astype(np.uint8)

        tp = np.logical_and(y_pred == 1, y_true == 1).sum()
        fp = np.logical_and(y_pred == 1, y_true == 0).sum()
        fn = np.logical_and(y_pred == 0, y_true == 1).sum()

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0
        candidate = (f1, recall, -threshold)

        if best_candidate is None or candidate > best_candidate:
            best_candidate = candidate
            best_threshold = float(threshold)

    return best_threshold

def indicator_row_to_skills(row, skill_names):
    """Converts a binary indicator row back into a list of skill names based on the provided skill names."""
    return [skill_names[j] for j, flag in enumerate(row) if flag == 1]

## Join Embeddings to Labels

This section connects the embedding files to the CSV labels.

This turns the raw embedding data into supervised learning data:
- the embedding file gives the feature vector
- the CSV gives the true skill list

After the join, each row contains:
- entity id
- title
- embedding
- actual skill list


In [45]:
skill_names = skills_embeddings["skill_name"].tolist()

jobs_train_df = attach_actual_skill_lists(
    jobs_embeddings[jobs_embeddings["split"] == "train"].copy(),
    job_labels,
    left_key="job_id",
    right_key="uuid",
)
jobs_val_df = attach_actual_skill_lists(
    jobs_embeddings[jobs_embeddings["split"] == "val"].copy(),
    job_labels,
    left_key="job_id",
    right_key="uuid",
)
jobs_test_df = attach_actual_skill_lists(
    jobs_embeddings[jobs_embeddings["split"] == "test"].copy(),
    job_labels,
    left_key="job_id",
    right_key="uuid",
)
courses_train_df = attach_actual_skill_lists(
    courses_embeddings[courses_embeddings["split"] == "train"].copy(),
    course_labels,
    left_key="course_code",
    right_key="moduleCode",
)
courses_val_df = attach_actual_skill_lists(
    courses_embeddings[courses_embeddings["split"] == "val"].copy(),
    course_labels,
    left_key="course_code",
    right_key="moduleCode",
)
courses_test_df = attach_actual_skill_lists(
    courses_embeddings[courses_embeddings["split"] == "test"].copy(),
    course_labels,
    left_key="course_code",
    right_key="moduleCode",
)

print("Supervised rows after joins:")
print(f"jobs train:    {len(jobs_train_df)}")
print(f"jobs val:      {len(jobs_val_df)}")
print(f"jobs test:     {len(jobs_test_df)}")
print(f"courses train: {len(courses_train_df)}")
print(f"courses val:   {len(courses_val_df)}")
print(f"courses test:  {len(courses_test_df)}")

Dropping 1 unlabeled rows for course_code: ['ACC4761E']
Supervised rows after joins:
jobs train:    59
jobs val:      19
jobs test:     20
courses train: 37
courses val:   13
courses test:  13


## Build Benchmark Variants

For the final OVR benchmark, this section uses the combined job + course training data.

It compares two thresholding strategies on the same `jobs_plus_courses` dataset:
- `global`: one shared probability cutoff across all skills
- `skill_specific`: one tuned cutoff per skill

This keeps the model focused on the jobs-and-curriculum gap table objective.


In [46]:
dataset_variants = {
    "jobs_plus_courses": {
        "train": pd.concat([jobs_train_df, courses_train_df], ignore_index=True),
        "val": pd.concat([jobs_val_df, courses_val_df], ignore_index=True),
        "test": pd.concat([jobs_test_df, courses_test_df], ignore_index=True),
    },
}

for variant_name, frames in dataset_variants.items():
    print(variant_name)
    print(f"  train rows: {len(frames['train'])}")
    print(f"  val rows:   {len(frames['val'])}")
    print(f"  test rows:  {len(frames['test'])}")

jobs_plus_courses
  train rows: 96
  val rows:   32
  test rows:  33


## Train, Tune, and Evaluate

This is the main experiment section.

For each dataset variant, this section:
- builds feature and label matrices
- trains one logistic regression model per skill
- tunes the threshold or thresholds on validation
- stores validation and test metrics for each threshold strategy
- chooses the final configuration using validation metrics only
- evaluates the locked predictions on test

It also stores a summary table and prediction previews so the model predictions can be inspected beyond the overall scores.


In [47]:
results = []
artifacts = {}

for variant_name, frames in dataset_variants.items():
    # The actual skill lists are already attached to the frames, so we just need to convert embeddings to numpy arrays and build indicator matrices for the skill labels.
    X_train = np.vstack(frames["train"]["embedding"].to_numpy()).astype(np.float32)
    X_val = np.vstack(frames["val"]["embedding"].to_numpy()).astype(np.float32)
    Y_train = build_indicator_matrix(frames["train"]["actual_skill_lists"].tolist(), skill_names)
    Y_val = build_indicator_matrix(frames["val"]["actual_skill_lists"].tolist(), skill_names)

    train_proba, val_proba, fitted_mask, fitted_models, skipped_skills, constant_proba = fit_ovr_probabilities(
        X_train,
        Y_train,
        X_val,
        skill_names,
    )

    # Compare a single shared threshold against per-skill thresholds on validation.
    global_threshold = tune_global_threshold(Y_val, val_proba)
    global_val_pred = (val_proba >= global_threshold).astype(np.uint8)
    global_val_micro_precision, global_val_micro_recall, global_val_micro_f1 = micro_metrics(Y_val, global_val_pred)
    global_val_macro_precision, global_val_macro_recall, global_val_macro_f1 = macro_metrics(Y_val, global_val_pred)

    results.append({
        "dataset_variant": variant_name,
        "threshold_type": "global",
        "val_micro_precision": global_val_micro_precision,
        "val_micro_recall": global_val_micro_recall,
        "val_micro_f1": global_val_micro_f1,
        "val_macro_precision": global_val_macro_precision,
        "val_macro_recall": global_val_macro_recall,
        "val_macro_f1": global_val_macro_f1,
        "global_threshold": global_threshold,
        "fitted_models": int(fitted_mask.sum()),
        "skipped_skills": len(skipped_skills),
        "train_rows": len(frames["train"]),
        "val_rows": len(frames["val"]),
        "test_rows": len(frames["test"]),
    })

    artifacts[(variant_name, "global")] = {
        "thresholds_df": pd.DataFrame({
            "skill_name": skill_names,
            "threshold": np.full(len(skill_names), global_threshold, dtype=np.float32),
            "was_fitted": fitted_mask,
            "train_positive_support": Y_train.sum(axis=0),
            "val_positive_support": Y_val.sum(axis=0),
        }),
        "frames": frames,
        "X_train": X_train,
        "X_val": X_val,
        "Y_train": Y_train,
        "Y_val": Y_val,
        "val_proba": val_proba,
        "fitted_models": fitted_models,
        "constant_proba": constant_proba,
        "fitted_mask": fitted_mask,
    }

    skill_thresholds = np.full(len(skill_names), 0.5, dtype=np.float32)
    # Only tune thresholds for skills with an actual fitted classifier.
    for j in np.where(fitted_mask)[0]:
        skill_thresholds[j] = tune_best_threshold_for_one_skill(Y_val[:, j], val_proba[:, j])

    skill_val_pred = (val_proba >= skill_thresholds[np.newaxis, :]).astype(np.uint8)
    skill_val_micro_precision, skill_val_micro_recall, skill_val_micro_f1 = micro_metrics(Y_val, skill_val_pred)
    skill_val_macro_precision, skill_val_macro_recall, skill_val_macro_f1 = macro_metrics(Y_val, skill_val_pred)

    results.append({
        "dataset_variant": variant_name,
        "threshold_type": "skill_specific",
        "val_micro_precision": skill_val_micro_precision,
        "val_micro_recall": skill_val_micro_recall,
        "val_micro_f1": skill_val_micro_f1,
        "val_macro_precision": skill_val_macro_precision,
        "val_macro_recall": skill_val_macro_recall,
        "val_macro_f1": skill_val_macro_f1,
        "global_threshold": np.nan,
        "fitted_models": int(fitted_mask.sum()),
        "skipped_skills": len(skipped_skills),
        "train_rows": len(frames["train"]),
        "val_rows": len(frames["val"]),
        "test_rows": len(frames["test"]),
    })

    artifacts[(variant_name, "skill_specific")] = {
        "thresholds_df": pd.DataFrame({
            "skill_name": skill_names,
            "threshold": skill_thresholds,
            "was_fitted": fitted_mask,
            "train_positive_support": Y_train.sum(axis=0),
            "val_positive_support": Y_val.sum(axis=0),
        }),
        "frames": frames,
        "X_train": X_train,
        "X_val": X_val,
        "Y_train": Y_train,
        "Y_val": Y_val,
        "val_proba": val_proba,
        "fitted_models": fitted_models,
        "constant_proba": constant_proba,
        "fitted_mask": fitted_mask,
    }

comparison_df = pd.DataFrame(results).sort_values(
    by=["val_micro_f1", "val_micro_recall", "val_macro_f1"],
    ascending=[False, False, False],
).reset_index(drop=True)
comparison_df

,dataset_variant,threshold_type,val_micro_precision,val_micro_recall,val_micro_f1,val_macro_precision,val_macro_recall,val_macro_f1,global_threshold,fitted_models,skipped_skills,train_rows,val_rows,test_rows
0,jobs_plus_courses,global,0.704225,0.684932,0.694444,0.066220,0.072410,0.066468,0.578499,74,155,96,32,33
1,jobs_plus_courses,skill_specific,0.086874,0.938356,0.159025,0.097555,0.134856,0.105305,NaN,74,155,96,32,33


## Inspect the Best Configuration

This section selects the strongest validation result from the comparison table and then shows the locked test metrics with a small prediction preview.

This helps check whether the selected model is making sensible skill predictions instead of relying only on summary metrics.


In [48]:
# For the best configuration based on validation performance, evaluate on the test set and prepare a detailed predictions DataFrame.
best_row = comparison_df.iloc[0]
best_key = (best_row["dataset_variant"], best_row["threshold_type"])
best_artifact = artifacts[best_key]
frames = best_artifact["frames"]

X_test = np.vstack(frames["test"]["embedding"].to_numpy()).astype(np.float32)
Y_test = build_indicator_matrix(frames["test"]["actual_skill_lists"].tolist(), skill_names)

# Use the fitted models and thresholds from the best configuration to predict on the test set and calculate metrics.
test_proba = predict_ovr_probabilities(
    X_test,
    best_artifact["fitted_models"],
    best_artifact["constant_proba"],
    skill_names,
)
# The thresholds are stored in the same order as the skill names, so we can directly use them to convert probabilities to binary predictions.
thresholds = best_artifact["thresholds_df"]["threshold"].to_numpy(dtype=np.float32)
test_pred = (test_proba >= thresholds[np.newaxis, :]).astype(np.uint8)

# Calculate micro and macro metrics on the test set using the predictions from the best model and thresholds.
test_micro_precision, test_micro_recall, test_micro_f1 = micro_metrics(Y_test, test_pred)
test_macro_precision, test_macro_recall, test_macro_f1 = macro_metrics(Y_test, test_pred)

test_metrics = pd.Series({
    "test_micro_precision": test_micro_precision,
    "test_micro_recall": test_micro_recall,
    "test_micro_f1": test_micro_f1,
    "test_macro_precision": test_macro_precision,
    "test_macro_recall": test_macro_recall,
    "test_macro_f1": test_macro_f1,
})

predicted_skill_lists = [indicator_row_to_skills(row, skill_names) for row in test_pred]
test_predictions_df = pd.DataFrame({
    "entity_type": frames["test"]["entity_type"],
    "entity_id": frames["test"]["entity_id"],
    "title": frames["test"]["display_title"],
    "actual_skills": [" | ".join(skills) for skills in frames["test"]["actual_skill_lists"]],
    "predicted_skills": [" | ".join(skills) for skills in predicted_skill_lists],
    "tp": [len(set(a) & set(p)) for a, p in zip(frames["test"]["actual_skill_lists"], predicted_skill_lists)],
    "fp": [len(set(p) - set(a)) for a, p in zip(frames["test"]["actual_skill_lists"], predicted_skill_lists)],
    "fn": [len(set(a) - set(p)) for a, p in zip(frames["test"]["actual_skill_lists"], predicted_skill_lists)],
})

best_artifact["X_test"] = X_test
best_artifact["Y_test"] = Y_test
best_artifact["test_proba"] = test_proba
best_artifact["test_pred"] = test_pred
best_artifact["test_metrics"] = test_metrics
best_artifact["predictions_df"] = test_predictions_df
best_artifact["thresholds_df"]["test_positive_support"] = Y_test.sum(axis=0)

selection_columns = [
    "dataset_variant",
    "threshold_type",
    "val_micro_precision",
    "val_micro_recall",
    "val_micro_f1",
    "val_macro_precision",
    "val_macro_recall",
    "val_macro_f1",
    "global_threshold",
    "fitted_models",
    "skipped_skills",
    "train_rows",
    "val_rows",
    "test_rows",
]

print("Selected jobs_plus_courses configuration based on validation performance:")
print(best_row[selection_columns].to_string())
print()
print("Locked test metrics for the selected configuration:")
print(test_metrics.to_string())
print()
print("Test-set prediction preview:")
print(test_predictions_df.head(10).to_string(index=False))


Selected jobs_plus_courses configuration based on validation performance:
dataset_variant        jobs_plus_courses
threshold_type                    global
val_micro_precision             0.704225
val_micro_recall                0.684932
val_micro_f1                    0.694444
val_macro_precision              0.06622
val_macro_recall                 0.07241
val_macro_f1                    0.066468
global_threshold                0.578499
fitted_models                         74
skipped_skills                       155
train_rows                            96
val_rows                              32
test_rows                             33

Locked test metrics for the selected configuration:
test_micro_precision    0.633094
test_micro_recall       0.606897
test_micro_f1           0.619718
test_macro_precision    0.049775
test_macro_recall       0.051410
test_macro_f1           0.048544

Test-set prediction preview:
entity_type                        entity_id                           

## Result Interpretation

This section focuses on result-level evidence rather than feature-level explanations. The model uses embedding dimensions as features, so individual feature names are not directly meaningful to a reader.

The model is interpreted using:
- the validation-selected model comparison, which explains why `jobs_plus_courses + global` or `jobs_plus_courses + skill_specific` was selected
- the selected model's test prediction preview, which shows concrete true positives, false positives, and false negatives
- the final ACC skills gap table, which links predicted job demand to curriculum coverage
- the data-quality diagnostics, which help explain why rare skills are less reliable

## Final ACC Skills Gap Table

This section refits the selected final ACC model on the full labeled dataset (`train + val + test`) and then scores all ACC jobs and ACC courses.

The predictions are used to create a rule-based table for high-demand skills based on:
- predicted job demand across all ACC jobs
- predicted curriculum coverage across all ACC courses
- whether any covering course is compulsory for the NUS Accountancy major


In [49]:
# These are the ACC module codes that are considered compulsory for the purpose of interpreting curriculum coverage of skills.
COMPULSORY_ACC_MODULE_CODES = {
    "ACC1701",
    "ACC2706",
    "ACC2707",
    "ACC2708",
    "ACC2709",
    "ACC2727",
    "ACC3701",
    "ACC3702",
    "ACC3703",
    "ACC3704",
    "ACC3705",
    "ACC3706",
    "ACC3707",
    "ACC3727",
}
HIGH_DEMAND_THRESHOLD_PERCENT = 10.0

def canonical_accountancy_module_code(module_code):
    """Converts a course code to a canonical form for matching against compulsory accountancy module codes. 
    Handles cases where the course code may have an additional trailing letter (e.g., "ACC1701A" -> "ACC1701")."""
    code = str(module_code).strip().upper()
    if code in COMPULSORY_ACC_MODULE_CODES:
        return code
    if code and code[-1].isalpha() and code[:-1] in COMPULSORY_ACC_MODULE_CODES:
        return code[:-1]
    return code

best_variant_name = str(best_row["dataset_variant"])
best_threshold_type = str(best_row["threshold_type"])
best_artifact = artifacts[best_key]

if "jobs_all_df" not in globals():
    jobs_all_df = pd.concat([
        jobs_train_df.assign(split="train"),
        jobs_val_df.assign(split="val"),
        jobs_test_df.assign(split="test"),
    ], ignore_index=True)

if "courses_all_df" not in globals():
    courses_all_df = pd.concat([
        courses_train_df.assign(split="train"),
        courses_val_df.assign(split="val"),
        courses_test_df.assign(split="test"),
    ], ignore_index=True)

full_fit_df = pd.concat([jobs_all_df, courses_all_df], ignore_index=True)

X_full_fit = np.vstack(full_fit_df["embedding"].to_numpy()).astype(np.float32)
Y_full_fit = build_indicator_matrix(full_fit_df["actual_skill_lists"].tolist(), skill_names)
X_jobs_all = np.vstack(jobs_all_df["embedding"].to_numpy()).astype(np.float32)
X_courses_all = np.vstack(courses_all_df["embedding"].to_numpy()).astype(np.float32)

# Prepare to refit models on the full ACC dataset for the selected configuration, and then use those models to predict probabilities on all job and course rows for the final gap analysis.
jobs_full_proba = np.zeros((X_jobs_all.shape[0], len(skill_names)), dtype=np.float32)
courses_full_proba = np.zeros((X_courses_all.shape[0], len(skill_names)), dtype=np.float32)
full_fitted_mask = np.zeros(len(skill_names), dtype=bool)
full_skipped_skills = []

# Refit after model selection so the final gap table uses all labeled ACC rows.
for j, skill_name in enumerate(skill_names):
    y_full_j = Y_full_fit[:, j]

    if len(np.unique(y_full_j)) < 2:
        full_skipped_skills.append(skill_name)
        continue

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42,
    )
    model.fit(X_full_fit, y_full_j)

    jobs_full_proba[:, j] = model.predict_proba(X_jobs_all)[:, 1]
    courses_full_proba[:, j] = model.predict_proba(X_courses_all)[:, 1]
    full_fitted_mask[j] = True

# Reuse the validation-selected thresholding rule when scoring the full ACC data.
if best_threshold_type == "global":
    selected_thresholds = np.full(len(skill_names), float(best_row["global_threshold"]), dtype=np.float32)
else:
    selected_thresholds = best_artifact["thresholds_df"]["threshold"].to_numpy(dtype=np.float32)

# Convert predicted probabilities to binary predictions using the selected thresholds, and then analyze the gaps between job demand and course coverage for each skill.
jobs_full_pred = (jobs_full_proba >= selected_thresholds[np.newaxis, :]).astype(np.uint8)
courses_full_pred = (courses_full_proba >= selected_thresholds[np.newaxis, :]).astype(np.uint8)

job_skill_counts = jobs_full_pred.sum(axis=0).astype(int)
course_skill_counts = courses_full_pred.sum(axis=0).astype(int)
job_denominator = max(len(jobs_all_df), 1)
course_codes = courses_all_df["entity_id"].astype(str).tolist()
course_titles = courses_all_df["display_title"].astype(str).tolist()

gap_rows = []
for j, skill_name in enumerate(skill_names):
    job_demand_percent = 100.0 * job_skill_counts[j] / job_denominator
    job_demand_label = "High" if job_demand_percent >= HIGH_DEMAND_THRESHOLD_PERCENT else "Low"

    course_idx = np.where(courses_full_pred[:, j] == 1)[0]
    compulsory_course_labels = []
    elective_course_labels = []

    for idx in course_idx:
        course_code = course_codes[idx]
        course_title = course_titles[idx]
        course_label = f"{course_code}: {course_title}"
        if canonical_accountancy_module_code(course_code) in COMPULSORY_ACC_MODULE_CODES:
            compulsory_course_labels.append(course_label)
        else:
            elective_course_labels.append(course_label)

    taught_in_any_course = len(course_idx) > 0
    taught_in_compulsory_course = len(compulsory_course_labels) > 0

    if taught_in_compulsory_course:
        curriculum_coverage = "Strong"
    elif taught_in_any_course:
        curriculum_coverage = "Weak"
    else:
        curriculum_coverage = "Very weak"

    # Gap labels combine predicted job demand with whether course coverage is compulsory.
    if job_demand_label == "High" and taught_in_compulsory_course:
        gap_interpretation = "Well covered"
    elif job_demand_label == "High" and taught_in_any_course:
        gap_interpretation = "Important gap - High demand skill but course not compulsory"
    elif job_demand_label == "High":
        gap_interpretation = "Critical gap - High demand skill not taught in ACC courses"
    elif taught_in_compulsory_course:
        gap_interpretation = "Compulsory coverage for lower-demand skill"
    elif taught_in_any_course:
        gap_interpretation = "Elective coverage for lower-demand skill"
    else:
        gap_interpretation = "Low demand and limited coverage"

    gap_rows.append({
        "skill_name": skill_name,
        "job_demand_percent": float(job_demand_percent),
        "job_demand": job_demand_label,
        "job_postings_with_skill": int(job_skill_counts[j]),
        "taught_in_any_course": "Yes" if taught_in_any_course else "No",
        "taught_in_compulsory_course": "Yes" if taught_in_compulsory_course else "No",
        "courses_covering_skill": int(course_skill_counts[j]),
        "curriculum_coverage": curriculum_coverage,
        "gap_interpretation": gap_interpretation,
        "compulsory_courses": " | ".join(compulsory_course_labels[:5]),
        "elective_courses": " | ".join(elective_course_labels[:5]),
    })

full_model_gap_table_df = pd.DataFrame(gap_rows).sort_values(
    by=["job_demand", "job_demand_percent", "curriculum_coverage", "skill_name"],
    ascending=[False, False, True, True],
).reset_index(drop=True)

high_demand_gap_table_df = full_model_gap_table_df.loc[
    full_model_gap_table_df["job_demand"] == "High"
].reset_index(drop=True)

final_model_refit_summary_df = pd.DataFrame({
    "selected_dataset_variant": [best_variant_name],
    "selected_threshold_type": [best_threshold_type],
    "full_fit_rows": [len(full_fit_df)],
    "jobs_scored": [len(jobs_all_df)],
    "courses_scored": [len(courses_all_df)],
    "fitted_skills_after_refit": [int(full_fitted_mask.sum())],
    "skipped_skills_after_refit": [len(full_skipped_skills)],
    "high_demand_threshold_percent": [HIGH_DEMAND_THRESHOLD_PERCENT],
})

print("Final ACC model refit summary:")
display(final_model_refit_summary_df)
print()
print("High-demand ACC skills gap table:")
display(high_demand_gap_table_df[[
    "skill_name",
    "job_demand_percent",
    "job_demand",
    "taught_in_any_course",
    "taught_in_compulsory_course",
    "curriculum_coverage",
    "gap_interpretation",
    "compulsory_courses",
    "elective_courses",
]])

Final ACC model refit summary:


,selected_dataset_variant,selected_threshold_type,full_fit_rows,jobs_scored,courses_scored,fitted_skills_after_refit,skipped_skills_after_refit,high_demand_threshold_percent
0,jobs_plus_courses,global,161,98,63,81,148,10.0



High-demand ACC skills gap table:


,skill_name,job_demand_percent,job_demand,taught_in_any_course,taught_in_compulsory_course,curriculum_coverage,gap_interpretation,compulsory_courses,elective_courses
0,Accounting Standards,75.510204,High,No,No,Very weak,Critical gap - High demand skill not taught in...,,
1,Transactional Accounting,70.408163,High,No,No,Very weak,Critical gap - High demand skill not taught in...,,
2,Financial Transactions,67.346939,High,No,No,Very weak,Critical gap - High demand skill not taught in...,,
3,Financial Closing,63.265306,High,No,No,Very weak,Critical gap - High demand skill not taught in...,,
4,Financial Reporting,62.244898,High,Yes,Yes,Strong,Well covered,ACC1701C: Accounting for Decision Makers | ACC...,ACC1701XB: Accounting for Decision Makers | AC...
5,Financial Administration,46.938776,High,No,No,Very weak,Critical gap - High demand skill not taught in...,,
6,Tax Compliance,42.857143,High,No,No,Very weak,Critical gap - High demand skill not taught in...,,
7,Audit Compliance,32.653061,High,No,No,Very weak,Critical gap - High demand skill not taught in...,,
8,Regulatory Compliance,29.591837,High,No,No,Very weak,Critical gap - High demand skill not taught in...,,
9,Accounting and Tax Systems,24.489796,High,No,No,Very weak,Critical gap - High demand skill not taught in...,,


## Summary

The selected configuration is `jobs_plus_courses + global`, chosen using validation performance. It achieves validation micro-F1 of `0.694` and locked test micro-F1 of `0.620`. The test macro-F1 is much lower at `0.049`, which shows that performance is mainly driven by common skills while rare skills remain difficult to model.

The final ACC gap table identifies `19` high-demand skills using the `10%` job-demand threshold. Only `4` of these high-demand skills are marked as well covered by compulsory courses: `Financial Reporting`, `Internal Controls`, `Auditing and Assurance Standards`, and `Financial Statements Review`.

The remaining `15` high-demand skills are marked as critical gaps because they are predicted in job postings but not predicted as taught in the ACC course set. The largest critical gaps are `Accounting Standards`, `Transactional Accounting`, `Financial Transactions`, `Financial Closing`, `Financial Administration`, and `Tax Compliance`.

The data-quality diagnostics explain why the model requires careful interpretation. Out of `229` total skills, only `74` skills were fitted in the selected validation model and `155` were skipped because the training data did not contain enough class variation. There are also `195` skills with best-model train support of at most `2`, so rare-skill predictions are much less reliable than common-skill predictions.

### Caveats and Limitations
- This benchmark assumes the CSV `extracted_skills` labels are the ground truth.
- Many skills are rare, so the model learns much better for common skills than rare ones.
- OVR cannot train a classifier for a skill if the train split does not contain both positive and negative examples for that skill.
- Micro-F1 is driven more by common skills, so it can look stronger than macro-F1.
- The skill-specific-threshold variant is more flexible, but it can overfit when the validation set is small.
- One course, `ACC4761E`, is excluded because it currently has no labeled `extracted_skills` value.


## Appendix: Data Quality Diagnostics

This section is kept as supporting material for the limitations. It checks whether the labels and dataset construction are affecting model performance.

The appendix audits four areas:
- overall skill coverage and rare-skill support
- skills that are out of vocabulary relative to the skill embedding file
- title-level label consistency and nearest-neighbor disagreement between similar jobs
- whether course labels look aligned enough with job labels to justify mixing them

These diagnostics are heuristic checks, so they are treated as quality-review tools rather than ground-truth error detectors.


### Skill Coverage and Rare-Skill Support

This cell checks how much training support each skill has. It helps explain why macro-F1 is low and why many skills are harder to predict: if a skill has little or no support in the training data, the OVR model has limited evidence to learn that class.

The OOV checks also confirm whether the labeled skills are present in the ACC skill embedding file. If OOV counts are zero, the main issue is not missing skill embeddings but limited label support for many skills.


In [50]:
from sklearn.metrics.pairwise import cosine_similarity

best_variant_name = str(best_row["dataset_variant"])
best_artifact = artifacts[best_key]
best_thresholds_df = best_artifact["thresholds_df"].reset_index(drop=True)
best_fitted_mask = np.asarray(best_artifact["fitted_mask"], dtype=bool)

jobs_all_df = pd.concat([
    jobs_train_df.assign(split="train"),
    jobs_val_df.assign(split="val"),
    jobs_test_df.assign(split="test"),
], ignore_index=True)

courses_all_df = pd.concat([
    courses_train_df.assign(split="train"),
    courses_val_df.assign(split="val"),
    courses_test_df.assign(split="test"),
], ignore_index=True)

def flatten_skill_lists(skill_lists):
    """Flattens a list of skill lists into a single list of individual skills."""
    return [skill for skills in skill_lists for skill in skills]

def canonical_skill_signature(skills):
    """Converts a list of skills into a canonical string signature by sorting and deduplicating the skills, then joining them with a pipe character."""
    return " | ".join(sorted(set(skills)))

def jaccard_score(left_skills, right_skills):
    """Calculates the Jaccard similarity score between two lists of skills by converting them to sets and computing the size of their intersection divided by the size of their union. Returns 1.0 if both sets are empty."""
    left_set = set(left_skills)
    right_set = set(right_skills)
    union = left_set | right_set
    if not union:
        return 1.0
    return len(left_set & right_set) / len(union)

def support_frame(df, prefix):
    """Builds a support DataFrame for the given embeddings DataFrame by creating a binary indicator matrix for the actual skill lists and summing the support for each skill."""
    indicator = build_indicator_matrix(df["actual_skill_lists"].tolist(), skill_names)
    return pd.DataFrame({
        "skill_name": skill_names,
        f"{prefix}_support": indicator.sum(axis=0).astype(int),
    })

# Build one support table so rare skills can be checked across every split.
support_df = pd.DataFrame({"skill_name": skill_names})
for prefix, frame in [
    ("jobs_train", jobs_train_df),
    ("jobs_val", jobs_val_df),
    ("jobs_test", jobs_test_df),
    ("courses_train", courses_train_df),
    ("courses_val", courses_val_df),
    ("courses_test", courses_test_df),
]:
    support_df = support_df.merge(support_frame(frame, prefix), on="skill_name", how="left")

support_cols = [col for col in support_df.columns if col.endswith("_support")]
support_df[support_cols] = support_df[support_cols].fillna(0).astype(int)
support_df["jobs_total_support"] = support_df[["jobs_train_support", "jobs_val_support", "jobs_test_support"]].sum(axis=1)
support_df["courses_total_support"] = support_df[["courses_train_support", "courses_val_support", "courses_test_support"]].sum(axis=1)
support_df["combined_train_support"] = support_df["jobs_train_support"] + support_df["courses_train_support"]
support_df["best_model_train_support"] = support_df["combined_train_support"]
support_df["best_model_was_fitted"] = best_fitted_mask
support_df["best_model_status"] = np.where(best_fitted_mask, "fitted", "skipped")
support_df["best_model_threshold"] = best_thresholds_df["threshold"].astype(float).to_numpy()

job_labels_audit_df = job_labels.copy()
course_labels_audit_df = course_labels.copy()
job_labels_audit_df["parsed_skill_lists"] = job_labels_audit_df["extracted_skills"].apply(parse_skill_list)
course_labels_audit_df["parsed_skill_lists"] = course_labels_audit_df["extracted_skills"].apply(parse_skill_list)

known_skill_set = set(skill_names)
job_oov_skills = sorted(set(flatten_skill_lists(job_labels_audit_df["parsed_skill_lists"])) - known_skill_set)
course_oov_skills = sorted(set(flatten_skill_lists(course_labels_audit_df["parsed_skill_lists"])) - known_skill_set)

summary_df = pd.DataFrame({
    "best_model_variant": [best_variant_name],
    "best_threshold_type": [str(best_row["threshold_type"])],
    "total_skills": [len(skill_names)],
    "best_model_fitted_skills": [int(best_fitted_mask.sum())],
    "best_model_skipped_skills": [int((~best_fitted_mask).sum())],
    "skills_with_zero_job_train_support": [int((support_df["jobs_train_support"] == 0).sum())],
    "skills_with_zero_combined_train_support": [int((support_df["combined_train_support"] == 0).sum())],
    "rare_skills_best_model_train_support_le_2": [int((support_df["best_model_train_support"] <= 2).sum())],
    "job_oov_skills": [len(job_oov_skills)],
    "course_oov_skills": [len(course_oov_skills)],
})

rare_skills_df = support_df.loc[
    support_df["best_model_train_support"] <= 2,
    [
        "skill_name",
        "jobs_train_support",
        "courses_train_support",
        "combined_train_support",
        "best_model_train_support",
        "best_model_status",
        "best_model_threshold",
    ],
].sort_values(
    by=["best_model_train_support", "combined_train_support", "skill_name"],
    ascending=[True, True, True],
).reset_index(drop=True)

print("Data coverage summary:")
display(summary_df)
print()
print("Rare-skill audit for the best model (train support <= 2):")
display(rare_skills_df.head(30))
print()
print(f"Job OOV skills relative to acc_skills_embeddings.jsonl: {len(job_oov_skills)}")
print(job_oov_skills[:25])
print()
print(f"Course OOV skills relative to acc_skills_embeddings.jsonl: {len(course_oov_skills)}")
print(course_oov_skills[:25])


Data coverage summary:


,best_model_variant,best_threshold_type,total_skills,best_model_fitted_skills,best_model_skipped_skills,skills_with_zero_job_train_support,skills_with_zero_combined_train_support,rare_skills_best_model_train_support_le_2,job_oov_skills,course_oov_skills
0,jobs_plus_courses,global,229,74,155,177,155,195,0,0



Rare-skill audit for the best model (train support <= 2):


,skill_name,jobs_train_support,courses_train_support,combined_train_support,best_model_train_support,best_model_status,best_model_threshold
0,Account Management,0,0,0,0,skipped,0.578499
1,Analytics and Computational Modelling,0,0,0,0,skipped,0.578499
2,Asset and Liability Management,0,0,0,0,skipped,0.578499
3,Attribution Analysis,0,0,0,0,skipped,0.578499
4,Audit Frameworks,0,0,0,0,skipped,0.578499
5,Auditor Independence,0,0,0,0,skipped,0.578499
6,Behavioural Finance,0,0,0,0,skipped,0.578499
7,Benchmarking,0,0,0,0,skipped,0.578499
8,Block Trading,0,0,0,0,skipped,0.578499
9,Book Building,0,0,0,0,skipped,0.578499



Job OOV skills relative to acc_skills_embeddings.jsonl: 0
[]

Course OOV skills relative to acc_skills_embeddings.jsonl: 0
[]


### Label Consistency Checks

This cell checks whether similar job records have consistent skill labels. Repeated job titles with different skill signatures can indicate noisy or inconsistent labels, while nearest-neighbor comparisons check whether highly similar jobs tend to share similar skills.

This supports the limitations discussion because inconsistent labels can reduce model performance even when the model architecture is reasonable.


In [51]:
job_labels_audit_df["normalized_title"] = (
    job_labels_audit_df["title"].fillna(job_labels_audit_df["raw_title"]).fillna("")
    .str.lower()
    .str.strip()
)
job_labels_audit_df["skill_signature"] = job_labels_audit_df["parsed_skill_lists"].apply(canonical_skill_signature)

title_conflicts_df = (
    job_labels_audit_df.groupby("normalized_title")
    .agg(
        num_rows=("uuid", "size"),
        distinct_skill_signatures=("skill_signature", "nunique"),
        sample_title=("title", lambda s: next((str(v) for v in s if pd.notna(v) and str(v).strip()), "")),
    )
    .reset_index()
)

title_conflicts_df = title_conflicts_df.loc[
    (title_conflicts_df["num_rows"] > 1) & (title_conflicts_df["distinct_skill_signatures"] > 1)
].sort_values(
    by=["distinct_skill_signatures", "num_rows", "sample_title"],
    ascending=[False, False, True],
).reset_index(drop=True)

jobs_matrix = np.vstack(jobs_all_df["embedding"].to_numpy()).astype(np.float32)
jobs_similarity = cosine_similarity(jobs_matrix)
# Exclude the row itself before selecting each job's nearest neighbor.
np.fill_diagonal(jobs_similarity, -np.inf)
job_skill_sets = jobs_all_df["actual_skill_lists"].apply(lambda skills: set(skills)).tolist()

nearest_neighbor_rows = []
for row_idx in range(len(jobs_all_df)):
    neighbor_idx = int(np.argmax(jobs_similarity[row_idx]))
    left_skills = job_skill_sets[row_idx]
    right_skills = job_skill_sets[neighbor_idx]
    nearest_neighbor_rows.append({
        "split": jobs_all_df.iloc[row_idx]["split"],
        "title": jobs_all_df.iloc[row_idx]["display_title"],
        "nearest_neighbor_title": jobs_all_df.iloc[neighbor_idx]["display_title"],
        "cosine_similarity": float(jobs_similarity[row_idx, neighbor_idx]),
        "label_jaccard": float(jaccard_score(left_skills, right_skills)),
        "num_shared_skills": int(len(left_skills & right_skills)),
        "current_skills": " | ".join(sorted(left_skills)),
        "neighbor_skills": " | ".join(sorted(right_skills)),
    })

nearest_neighbor_df = pd.DataFrame(nearest_neighbor_rows)
# Focus the manual review on the most similar quartile of job pairs.
similarity_threshold = float(nearest_neighbor_df["cosine_similarity"].quantile(0.75))
high_similarity_disagreement_df = nearest_neighbor_df.loc[
    (nearest_neighbor_df["cosine_similarity"] >= similarity_threshold)
    & (nearest_neighbor_df["label_jaccard"] < 0.25)
].sort_values(
    by=["label_jaccard", "cosine_similarity"],
    ascending=[True, False],
).reset_index(drop=True)

consistency_summary_df = pd.DataFrame({
    "same_title_conflict_groups": [len(title_conflicts_df)],
    "median_nearest_job_cosine_similarity": [nearest_neighbor_df["cosine_similarity"].median()],
    "median_nearest_job_label_jaccard": [nearest_neighbor_df["label_jaccard"].median()],
    "high_similarity_threshold": [similarity_threshold],
    "high_similarity_jobs_with_label_jaccard_below_0_25": [len(high_similarity_disagreement_df)],
})

print("Label consistency summary:")
display(consistency_summary_df)
print()
print("Repeated job titles with conflicting skill signatures (top 10):")
display(title_conflicts_df.head(10))
print()
if high_similarity_disagreement_df.empty:
    print("No high-similarity job pairs had label Jaccard below 0.25.")
else:
    print("High-similarity job pairs with low label overlap (top 5):")
    display(high_similarity_disagreement_df.head(5))


Label consistency summary:


,same_title_conflict_groups,median_nearest_job_cosine_similarity,median_nearest_job_label_jaccard,high_similarity_threshold,high_similarity_jobs_with_label_jaccard_below_0_25
0,6,0.897267,0.522727,0.983182,0



Repeated job titles with conflicting skill signatures (top 10):


,normalized_title,num_rows,distinct_skill_signatures,sample_title
0,accounts assistant,10,9,ACCOUNTS ASSISTANT
1,audit associate,6,6,Audit Associate
2,accounts executive,3,3,Accounts Executive
3,6 months accounts payable executive #njn,2,2,6 Months Accounts Payable Executive #NJN
4,account executive,2,2,account executive
5,audit assistant,2,2,audit assistant



No high-similarity job pairs had label Jaccard below 0.25.


### Jobs and Courses Alignment

This cell compares how skills appear in job records versus course records. It checks whether the combined jobs-plus-courses training setup is reasonably aligned and highlights skills that appear much more often in jobs than in courses.

This directly supports the skills gap interpretation, because large job-course frequency mismatches help explain why some high-demand skills are flagged as curriculum gaps.


In [52]:
alignment_df = support_df[["skill_name", "jobs_train_support", "courses_train_support"]].copy()
alignment_df = alignment_df.loc[(alignment_df["jobs_train_support"] + alignment_df["courses_train_support"]) > 0].copy()
alignment_df["jobs_train_frequency"] = alignment_df["jobs_train_support"] / max(len(jobs_train_df), 1)
alignment_df["courses_train_frequency"] = alignment_df["courses_train_support"] / max(len(courses_train_df), 1)
# Positive gaps mean a skill appears more often in jobs than courses.
alignment_df["frequency_gap"] = alignment_df["jobs_train_frequency"] - alignment_df["courses_train_frequency"]
alignment_df["abs_frequency_gap"] = alignment_df["frequency_gap"].abs()

shared_train_skills = int(((alignment_df["jobs_train_support"] > 0) & (alignment_df["courses_train_support"] > 0)).sum())
job_only_train_skills = int(((alignment_df["jobs_train_support"] > 0) & (alignment_df["courses_train_support"] == 0)).sum())
course_only_train_skills = int(((alignment_df["jobs_train_support"] == 0) & (alignment_df["courses_train_support"] > 0)).sum())
frequency_correlation = alignment_df["jobs_train_frequency"].corr(alignment_df["courses_train_frequency"])

jobs_avg_skills = jobs_all_df["actual_skill_lists"].apply(len).mean()
courses_avg_skills = courses_all_df["actual_skill_lists"].apply(len).mean()

courses_matrix = np.vstack(courses_all_df["embedding"].to_numpy()).astype(np.float32)
course_to_job_similarity = cosine_similarity(courses_matrix, jobs_matrix)
# Nearest job matches are only a diagnostic for label alignment, not training labels.
best_job_match_idx = course_to_job_similarity.argmax(axis=1)

course_job_alignment_rows = []
for course_idx, job_idx in enumerate(best_job_match_idx):
    course_skills = set(courses_all_df.iloc[course_idx]["actual_skill_lists"])
    job_skills = job_skill_sets[job_idx]
    course_job_alignment_rows.append({
        "course_code": courses_all_df.iloc[course_idx]["entity_id"],
        "course_title": courses_all_df.iloc[course_idx]["display_title"],
        "matched_job_title": jobs_all_df.iloc[job_idx]["display_title"],
        "cosine_similarity": float(course_to_job_similarity[course_idx, job_idx]),
        "label_jaccard": float(jaccard_score(course_skills, job_skills)),
    })

course_job_alignment_df = pd.DataFrame(course_job_alignment_rows).sort_values(
    by=["label_jaccard", "cosine_similarity"],
    ascending=[True, False],
).reset_index(drop=True)

low_overlap_course_matches_df = course_job_alignment_df.loc[
    course_job_alignment_df["label_jaccard"] < 0.10
].reset_index(drop=True)

alignment_summary_df = pd.DataFrame({
    "shared_train_skills": [shared_train_skills],
    "job_only_train_skills": [job_only_train_skills],
    "course_only_train_skills": [course_only_train_skills],
    "jobs_vs_courses_train_frequency_corr": [frequency_correlation],
    "avg_skills_per_job_record": [jobs_avg_skills],
    "avg_skills_per_course_record": [courses_avg_skills],
    "median_course_to_nearest_job_cosine": [course_job_alignment_df["cosine_similarity"].median()],
    "median_course_to_nearest_job_label_jaccard": [course_job_alignment_df["label_jaccard"].median()],
    "courses_with_nearest_job_label_jaccard_below_0_10": [len(low_overlap_course_matches_df)],
})

largest_frequency_gap_df = alignment_df.sort_values(
    by=["abs_frequency_gap", "skill_name"],
    ascending=[False, True],
).reset_index(drop=True)

frequency_gap_columns = [
    "skill_name",
    "jobs_train_support",
    "courses_train_support",
    "jobs_train_frequency",
    "courses_train_frequency",
    "frequency_gap",
]

print("Jobs vs courses alignment summary:")
display(alignment_summary_df)
print()
print("Skills with the largest train-frequency mismatch between jobs and courses (top 10):")
display(largest_frequency_gap_df[frequency_gap_columns].head(10))
print()
if low_overlap_course_matches_df.empty:
    print("No course had nearest-job label overlap below 0.10.")
else:
    print("Courses whose nearest job neighbor has very low label overlap (top 5):")
    display(low_overlap_course_matches_df.head(5))


Jobs vs courses alignment summary:


,shared_train_skills,job_only_train_skills,course_only_train_skills,jobs_vs_courses_train_frequency_corr,avg_skills_per_job_record,avg_skills_per_course_record,median_course_to_nearest_job_cosine,median_course_to_nearest_job_label_jaccard,courses_with_nearest_job_label_jaccard_below_0_10
0,19,33,22,0.53416,6.836735,2.428571,0.670055,0.125,25



Skills with the largest train-frequency mismatch between jobs and courses (top 10):


,skill_name,jobs_train_support,courses_train_support,jobs_train_frequency,courses_train_frequency,frequency_gap
0,Transactional Accounting,33,1,0.559322,0.027027,0.532295
1,Financial Closing,31,0,0.525424,0.000000,0.525424
2,Financial Transactions,34,2,0.576271,0.054054,0.522217
3,Financial Reporting,41,9,0.694915,0.243243,0.451672
4,Tax Compliance,25,0,0.423729,0.000000,0.423729
5,Accounting Standards,42,11,0.711864,0.297297,0.414567
6,Audit Compliance,18,0,0.305085,0.000000,0.305085
7,Regulatory Compliance,18,1,0.305085,0.027027,0.278058
8,Financial Administration,15,0,0.254237,0.000000,0.254237
9,Engagement Execution,14,0,0.237288,0.000000,0.237288



Courses whose nearest job neighbor has very low label overlap (top 5):


,course_code,course_title,matched_job_title,cosine_similarity,label_jaccard
0,ACC4714,Advanced and Sustainability Assurance,Audit Associate/Senior,0.729057,0.0
1,ACC4761A,Seminars in Accounting: Internal Audit,Audit Assistant,0.688501,0.0
2,ACC4712,Forensic Accounting,Senior Audit Associate/Supervisor,0.670896,0.0
3,ACC4613,Forensic Accounting,Senior Audit Associate/Supervisor,0.670134,0.0
4,ACC4761H,Sem. in Acctg: Accounting & Business Analysis ...,Accounts Assistant (5 days / Marine Parade),0.668700,0.0
